In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# LAMMPS LOG PARSER
# ============================================================

def extract_thermo_data(
    log_path,
    cutoff_step=None,
    last_ns=None,
    timestep_fs=1.0,
):
    """
    Extract thermo quantities from a LAMMPS log file.

    Parameters
    ----------
    log_path : str
        Path to log.lammps

    cutoff_step : int or None
        Use data after this step only

    last_ns : float or None
        Extract only last X ns

    timestep_fs : float
        MD timestep in fs

    Returns
    -------
    thermo_dict : dict
        Dictionary containing all thermo quantities
    """

    with open(log_path, "r") as f:
        lines = f.readlines()

    # --------------------------------------------------------
    # Find thermo header
    # --------------------------------------------------------

    header = None
    start_idx = None

    for i, line in enumerate(lines):
        if line.strip().startswith("Step"):
            cols = line.split()

            if "PotEng" in cols or "pe" in cols:
                header = cols
                start_idx = i
                break

    if header is None:
        raise RuntimeError(f"Thermo header not found in {log_path}")

    # --------------------------------------------------------
    # Store thermo data
    # --------------------------------------------------------

    thermo = {key: [] for key in header}

    for line in lines[start_idx + 1:]:

        parts = line.split()

        if len(parts) != len(header):
            continue

        try:
            step = int(parts[0])
        except:
            continue

        for key, value in zip(header, parts):

            try:
                thermo[key].append(float(value))
            except:
                pass

    # Convert to numpy arrays
    for key in thermo:
        thermo[key] = np.array(thermo[key])

    # --------------------------------------------------------
    # Filtering
    # --------------------------------------------------------

    steps = thermo["Step"]

    if cutoff_step is not None:

        mask = steps >= cutoff_step

    elif last_ns is not None:

        max_step = steps.max()

        steps_target = int((last_ns * 1e6) / timestep_fs)

        cutoff = max_step - steps_target

        mask = steps >= cutoff

    else:

        mask = np.ones_like(steps, dtype=bool)

    for key in thermo:
        thermo[key] = thermo[key][mask]

    return thermo


# ============================================================
# ENERGY ANALYSIS
# ============================================================

def compute_adsorption_energy(
    close_log,
    peg_log,
    csh_log,
    cutoff_step=None,
    average_last_n=None,
    two_box=False,
):
    """
    Compute adsorption energy:

    E_ads = E_close - (E_peg + E_csh)
    """

    close_data = extract_thermo_data(
        close_log,
        cutoff_step=cutoff_step,
    )

    peg_data = extract_thermo_data(
        peg_log,
        cutoff_step=cutoff_step,
    )

    csh_data = extract_thermo_data(
        csh_log,
        cutoff_step=cutoff_step,
    )

    # --------------------------------------------------------
    # Detect PE column
    # --------------------------------------------------------

    pe_key = "PotEng" if "PotEng" in close_data else "pe"

    close_pe = close_data[pe_key]
    peg_pe = peg_data[pe_key]
    csh_pe = csh_data[pe_key]

    # --------------------------------------------------------
    # Average strategy
    # --------------------------------------------------------

    if average_last_n is not None:

        E_close = np.mean(close_pe[-average_last_n:])
        E_peg = np.mean(peg_pe[-average_last_n:])
        E_csh = np.mean(csh_pe[-average_last_n:])

    else:

        E_close = np.mean(close_pe)
        E_peg = np.mean(peg_pe)
        E_csh = np.mean(csh_pe)

    if two_box:
        E_ads = E_close - E_peg
    else:
        E_ads = E_close - (E_peg + E_csh)

    # --------------------------------------------------------
    # Print summary
    # --------------------------------------------------------

    print("\n==============================")
    print("Adsorption Energy Summary")
    print("==============================")

    print(f"Close System     : {E_close:.6f}")
    print(f"PEG + Water      : {E_peg:.6f}")
    print(f"CSH + Water      : {E_csh:.6f}")
    print(f"Separated System : {E_peg + E_csh:.6f}")

    print("------------------------------")
    print(f"Adsorption Energy: {E_ads:.6f} kcal/mol")
    print("==============================\n")

    return {
        "close": close_data,
        "peg": peg_data,
        "csh": csh_data,
        "E_ads": E_ads,
    }


# ============================================================
# PLOTTING
# ============================================================

def plot_thermo_grid(data_dict):

    quantities = [
        "Temp",
        "PotEng",
        "TotEng",
        "Press",
    ]

    systems = [
        ("close", "Close"),
        ("peg", "PEG + Water"),
        ("csh", "CSH + Water"),
    ]

    fig, axs = plt.subplots(
        len(quantities),
        len(systems),
        figsize=(15, 12),
    )

    for row, quantity in enumerate(quantities):

        for col, (key, title) in enumerate(systems):

            ax = axs[row, col]

            data = data_dict[key]

            if quantity not in data:
                ax.set_visible(False)
                continue

            ax.plot(data[quantity])

            ax.set_title(f"{title} : {quantity}")

            ax.set_xlabel("Frame")
            ax.set_ylabel(quantity)

    plt.tight_layout()
    plt.show()


def plot_thermo_grid_together(data_dict, quantities, systems, fig, axs, name=None):

    for row, quantity in enumerate(quantities):

        for col, (key, title) in enumerate(systems):

            ax = axs[row, col]

            data = data_dict[key]

            if quantity not in data:
                ax.set_visible(False)
                continue

            ax.plot(data[quantity], label=name)

            ax.set_title(f"{title} : {quantity}")

            ax.legend()
            ax.set_xlabel("Frame")
            ax.set_ylabel(quantity)

# ============================================================
# USER INPUT
# ============================================================

BASE = Path(
    ""
    "MDSetup/example/mechanical_properties/"
    "0_CSH_transfer/CSH_surface/try/"
    "2_combined_new"
)

RUN = "0_run8"

close_log = BASE / RUN / "1_all3/log.lammps"
peg_log   = BASE / RUN / "2_peg/log.lammps"
csh_log   = BASE / RUN / "3_csh_water/log.lammps"


# ============================================================
# RUN ANALYSIS
# ============================================================

results = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)
results1 = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=None)


# ============================================================
# PLOT
# ============================================================

plot_thermo_grid(results)

In [ ]:

RUN = "0_run8"

close_log = BASE / RUN / "1_all3/log.lammps"
peg_log   = BASE / RUN / "2_peg/log.lammps"
csh_log   = BASE / RUN / "3_csh_water/log.lammps"


# ============================================================
# RUN ANALYSIS
# ============================================================

results = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


# ============================================================
# PLOT
# ============================================================

plot_thermo_grid(results)

In [ ]:

RUN = "0_run8"

close_log = BASE / RUN / "1_all3/new_3ns_fixcom/log.lammps"
peg_log   = BASE / RUN / "2_peg/log.lammps"
csh_log   = BASE / RUN / "3_csh_water/new_3ns_fixcom/log.lammps"

# ============================================================
# RUN ANALYSIS
# ============================================================

results1 = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


# ============================================================
# PLOT
# ============================================================

plot_thermo_grid(results1)

In [ ]:
plt.plot(results1['close']['PotEng'][:],'r')
plt.plot(results1['peg']['PotEng'][:]+results1['csh']['PotEng'][:],'b')

In [ ]:
plt.plot(results['close']['PotEng'][-1500:],'r')
plt.plot(results['peg']['PotEng'][-1500:]+results['csh']['PotEng'][-1500:],'b')

In [ ]:
plt.plot(results['close']['PotEng'][-1000:],'r')
plt.plot(results['peg']['PotEng'][-1000:]+results['csh']['PotEng'][-1000:],'b')

In [ ]:
RUN = "0_run9"

close_log = BASE / RUN / "1_1-12E/S1/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"

# ============================================================
# RUN ANALYSIS
# ============================================================

results_11 = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)

# ============================================================
# PLOT
# ============================================================

plot_thermo_grid(results_11)

In [ ]:
RUN = "0_run9"

close_log = BASE / RUN / "2_1-8E/S1/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"

# ============================================================
# RUN ANALYSIS
# ============================================================

results_21 = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


# ============================================================
# PLOT
# ============================================================

plot_thermo_grid(results_21)

In [ ]:
RUN = "0_run9"

close_log = BASE / RUN / "3_1-6E/S1/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"

# ============================================================
# RUN ANALYSIS
# ============================================================

results_31 = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


# ============================================================
# PLOT
# ============================================================

plot_thermo_grid(results_31)

In [ ]:
plt.plot(results_11['close']['Lz'][-3000:])
plt.plot(results_11['peg']['Lz'][-3000:]+results_11['csh']['Lz'][-3000:])

plt.plot(results_21['close']['Lz'][-3000:])
plt.plot(results_21['peg']['Lz'][-3000:]+results_21['csh']['Lz'][-3000:])

plt.plot(results_31['close']['Lz'][-3000:])
plt.plot(results_31['peg']['Lz'][-3000:]+results_31['csh']['Lz'][-3000:])



# print mean of all 6
print((results_11['close']['Lz'][-3000:]).mean())
print((results_21['close']['Lz'][-3000:]).mean())
print((results_31['close']['Lz'][-3000:]).mean())
print((results_11['peg']['Lz'][-3000:]+results_11['csh']['Lz'][-3000:]).mean())
print((results_21['peg']['Lz'][-3000:]+results_21['csh']['Lz'][-3000:]).mean())
print((results_31['peg']['Lz'][-3000:]+results_31['csh']['Lz'][-3000:]).mean())




print((results_11['peg']['Lz'][-3000:]).mean())
print((results_21['peg']['Lz'][-3000:]).mean())
print((results_31['peg']['Lz'][-3000:]).mean())



print((results_11['csh']['Lz'][-3000:]).mean())

In [ ]:
plt.plot(results_11['close']['PotEng'][:],'r')
plt.plot(results_11['peg']['PotEng'][-7512:]+results_11['csh']['PotEng'][-7512:],'b')


plt.plot(results1['close']['PotEng'][:])
plt.plot(results1['peg']['PotEng'][:]+results1['csh']['PotEng'][:])

In [ ]:
plt.plot(results_11['close']['PotEng'], c='r')
# plt.plot(results['peg']['PotEng'], c='g')
plt.plot(results_11['csh']['PotEng'], c='b')


In [ ]:
plt.plot(results['close']['PotEng'])
plt.plot(results1['close']['PotEng'])

In [ ]:
plt.plot(results['peg']['PotEng'])
plt.plot(results1['peg']['PotEng'])

In [ ]:
plt.plot(results['csh']['PotEng'])
plt.plot(results1['csh']['PotEng'])


In [ ]:

mask = results_31old['csh']['Press'][-3500:]<=300

fig, ax1 = plt.subplots()

ax1.plot(results_31old['csh']['Lz'][-3500:][mask], color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx() 
ax2.plot(results_31old['csh']['Press'][-3500:][mask], color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.show()


In [ ]:
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=-100].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=-20].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=-10].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=0].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=10].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=100].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=200].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=300].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=400].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=500].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=600].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=700].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=800].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=900].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1000].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1100].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1200].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1300].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1400].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1500].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1600].mean())
print(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1700].mean())


ll = [results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=-100].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=-20].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=-10].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=0].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=10].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=100].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=200].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=300].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=400].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=500].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=600].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=700].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=800].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=900].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1000].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1100].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1200].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1300].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1400].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1500].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1600].mean(), results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=1700].mean()]

plt.plot([-100, -20, -10, 0,10,100,200,300,400,500,600,700,800,900,1000,1100,1200,1300,1400,1500,1600,1700], ll)

In [ ]:
ll = []
for i in range(-1000, 3001, 10):
    ll.append(results_31old['close']['Lz'][-3500:][results_31old['close']['Press'][-3500:]<=i].mean())

plt.plot(range(-1000, 3001, 10), ll)
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()

In [ ]:
ll = []
for i in range(-1000, 3001, 10):
    ll.append(results_31old['peg']['Lz'][-3500:][results_31old['peg']['Press'][-3500:]<=i].mean())

plt.plot(range(-1000, 3001, 10), ll)
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()

In [ ]:
ll = []
for i in range(-1000, 1001, 10):
    ll.append(results_11old['peg']['Lz'][-3500:][(results_11old['peg']['Press'][-3500:]<=i) * (results_11old['peg']['Press'][-3500:]>=(i-100))].mean())

plt.plot(range(-1000, 1001, 10), ll, label='Mean (P±100)')
plt.axhline(results_11old['peg']['Lz'][-3500:].mean(), color='red', linestyle='--', label='Overall Mean')

plt.axhline(results_11old['peg']['Lz'][-3500:][(results_11old['peg']['Press'][-3500:]<=250) * (results_11old['peg']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--', label='Mean (0±250)')
plt.axhline(results_11old['peg']['Lz'][-3500:][(results_11old['peg']['Press'][-3500:]<=100) * (results_11old['peg']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--', label='Mean (0±100)')
plt.legend()
# plt.axhline(101.19978360666667, color='blue', linestyle='--')
plt.title("Lz vs Pressure (PEG) - 1:1-12E")
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.grid()
plt.show()

In [ ]:
ll = []
for i in range(-1000, 1001, 10):
    ll.append(results_21old['peg']['Lz'][-3500:][(results_21old['peg']['Press'][-3500:]<=i) * (results_21old['peg']['Press'][-3500:]>=(i-100))].mean())

plt.plot(range(-1000, 1001, 10), ll, label='Mean (P±100)')
plt.axhline(results_21old['peg']['Lz'][-3500:].mean(), color='red', linestyle='--', label='Overall Mean')

plt.axhline(results_21old['peg']['Lz'][-3500:][(results_21old['peg']['Press'][-3500:]<=250) * (results_21old['peg']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--', label='Mean (0±250)')
plt.axhline(results_21old['peg']['Lz'][-3500:][(results_21old['peg']['Press'][-3500:]<=100) * (results_21old['peg']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--', label='Mean (0±100)')
# plt.axhline(101.31636158999999, color='blue', linestyle='--')
plt.legend()
plt.title("Lz vs Pressure (PEG) - 2:1-8E")
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.grid()
plt.show()

In [ ]:
ll = []
for i in range(-1000, 1001, 10):
    ll.append(results_31old['peg']['Lz'][-3500:][(results_31old['peg']['Press'][-3500:]<=i) * (results_31old['peg']['Press'][-3500:]>=(i-100))].mean())

plt.plot(range(-1000, 1001, 10), ll, label='Mean (P±100)')
plt.axhline(results_31old['peg']['Lz'][-3500:].mean(), color='red', linestyle='--', label='Overall Mean')
plt.axhline(results_31old['peg']['Lz'][-3500:][(results_31old['peg']['Press'][-3500:]<=250) * (results_31old['peg']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--', label='Mean (0±250)')
plt.axhline(results_31old['peg']['Lz'][-3500:][(results_31old['peg']['Press'][-3500:]<=100) * (results_31old['peg']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--', label='Mean (0±100)')
# plt.axhline(101.40595109666667, color='blue', linestyle='--')

plt.legend()
plt.title("Lz vs Pressure (PEG) - 3:1-6E")
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.grid()
plt.show()



In [ ]:
ll = []
for i in range(-1000, 3001, 10):
    ll.append(results_31old['csh']['Lz'][-3500:][(results_31old['csh']['Press'][-3500:]<=i) * (results_31old['csh']['Press'][-3500:]>=(i-100))].mean())

plt.plot(range(-1000, 3001, 10), ll, label='Mean (P±100)')

plt.axhline(results_31old['csh']['Lz'][-3500:].mean(), color='red', linestyle='--', label='Overall Mean')
plt.axhline(results_31old['csh']['Lz'][-3500:][(results_31old['csh']['Press'][-3500:]<=250) * (results_31old['csh']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--', label='Mean (0±250)')
plt.axhline(results_31old['csh']['Lz'][-3500:][(results_31old['csh']['Press'][-3500:]<=100) * (results_31old['csh']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--', label='Mean (0±100)')
plt.axhline(58.12622, color='blue', linestyle='--', label='previously used Lz')

plt.legend()

plt.title("Lz vs Pressure (S3-CSH)")

plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.grid()
plt.show()

In [ ]:
plt.plot([101.19,101.21],[101.19,101.21])
plt.axhline(results_11old['peg']['Lz'][-3500:].mean(), color='red', linestyle='--')
plt.axhline(results_11old['peg']['Lz'][-3500:][(results_11old['peg']['Press'][-3500:]<=250) * (results_11old['peg']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--')
plt.axhline(results_11old['peg']['Lz'][-3500:][(results_11old['peg']['Press'][-3500:]<=100) * (results_11old['peg']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--')
plt.axhline(101.19978360666667, color='blue', linestyle='--')
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()


plt.plot([101.3,101.35],[101.3,101.35])
plt.axhline(results_21old['peg']['Lz'][-3500:].mean(), color='red', linestyle='--')
plt.axhline(results_21old['peg']['Lz'][-3500:][(results_21old['peg']['Press'][-3500:]<=250) * (results_21old['peg']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--')
plt.axhline(results_21old['peg']['Lz'][-3500:][(results_21old['peg']['Press'][-3500:]<=100) * (results_21old['peg']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--')
plt.axhline(101.31636158999999, color='blue', linestyle='--')
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()



plt.plot([101.38,101.42],[101.38,101.42])
plt.axhline(results_31old['peg']['Lz'][-3500:].mean(), color='red', linestyle='--')
plt.axhline(results_31old['peg']['Lz'][-3500:][(results_31old['peg']['Press'][-3500:]<=250) * (results_31old['peg']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--')
plt.axhline(results_31old['peg']['Lz'][-3500:][(results_31old['peg']['Press'][-3500:]<=100) * (results_31old['peg']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--')
plt.axhline(101.40595109666667, color='blue', linestyle='--')

plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()


plt.plot([58.1,58.2],[58.1,58.2])
plt.axhline(results_31old['csh']['Lz'][-3500:].mean(), color='red', linestyle='--')
plt.axhline(results_31old['csh']['Lz'][-3500:][(results_31old['csh']['Press'][-3500:]<=250) * (results_31old['csh']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--')
plt.axhline(results_31old['csh']['Lz'][-3500:][(results_31old['csh']['Press'][-3500:]<=100) * (results_31old['csh']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--')
plt.axhline(58.12622, color='blue', linestyle='--')
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()




In [ ]:

print('250')
print((results_11old['peg']['Lz'][-3500:][(results_11old['peg']['Press'][-3500:]<=250) * (results_11old['peg']['Press'][-3500:]>=-250)].mean()))
print((results_21old['peg']['Lz'][-3500:][(results_21old['peg']['Press'][-3500:]<=250) * (results_21old['peg']['Press'][-3500:]>=-250)].mean()))
print((results_31old['peg']['Lz'][-3500:][(results_31old['peg']['Press'][-3500:]<=250) * (results_31old['peg']['Press'][-3500:]>=-250)].mean()))
print((results_31old['csh']['Lz'][-3500:][(results_31old['csh']['Press'][-3500:]<=250) * (results_31old['csh']['Press'][-3500:]>=-250)].mean()))

print('100')
print((results_11old['peg']['Lz'][-3500:][(results_11old['peg']['Press'][-3500:]<=100) * (results_11old['peg']['Press'][-3500:]>=-100)].mean()))
print((results_21old['peg']['Lz'][-3500:][(results_21old['peg']['Press'][-3500:]<=100) * (results_21old['peg']['Press'][-3500:]>=-100)].mean()))
print((results_31old['peg']['Lz'][-3500:][(results_31old['peg']['Press'][-3500:]<=100) * (results_31old['peg']['Press'][-3500:]>=-100)].mean()))
print((results_31old['csh']['Lz'][-3500:][(results_31old['csh']['Press'][-3500:]<=100) * (results_31old['csh']['Press'][-3500:]>=-100)].mean()))

print('avg')
print(((results_11old['peg']['Lz'][-3500:][(results_11old['peg']['Press'][-3500:]<=250) * (results_11old['peg']['Press'][-3500:]>=-250)].mean()) + (results_11old['peg']['Lz'][-3500:][(results_11old['peg']['Press'][-3500:]<=100) * (results_11old['peg']['Press'][-3500:]>=-100)].mean()))/2)
print(((results_21old['peg']['Lz'][-3500:][(results_21old['peg']['Press'][-3500:]<=250) * (results_21old['peg']['Press'][-3500:]>=-250)].mean()) + (results_21old['peg']['Lz'][-3500:][(results_21old['peg']['Press'][-3500:]<=100) * (results_21old['peg']['Press'][-3500:]>=-100)].mean()))/2)
print(((results_31old['peg']['Lz'][-3500:][(results_31old['peg']['Press'][-3500:]<=250) * (results_31old['peg']['Press'][-3500:]>=-250)].mean()) + (results_31old['peg']['Lz'][-3500:][(results_31old['peg']['Press'][-3500:]<=100) * (results_31old['peg']['Press'][-3500:]>=-100)].mean()))/2)
print(((results_31old['csh']['Lz'][-3500:][(results_31old['csh']['Press'][-3500:]<=250) * (results_31old['csh']['Press'][-3500:]>=-250)].mean()) + (results_31old['csh']['Lz'][-3500:][(results_31old['csh']['Press'][-3500:]<=100) * (results_31old['csh']['Press'][-3500:]>=-100)].mean()))/2)


In [ ]:
ll = []
for i in range(-1000, 3001, 10):
    ll.append(results_31old['csh']['Lz'][-3500:][results_31old['csh']['Press'][-3500:]<=i].mean())

plt.plot(range(-1000, 3001, 10), ll)
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()

In [ ]:
RUN = "0_run9"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_11old = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_21old = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_31old = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11old, quantities, systems, fig, axs, name='1:1 (12E)')
plot_thermo_grid_together(results_21old, quantities, systems, fig, axs, name='2:1 (8E)')
plot_thermo_grid_together(results_31old, quantities, systems, fig, axs, name='3:1 (6E)')

plt.tight_layout()
plt.show()


In [ ]:
pe_npt1 = results_11old['close']['PotEng'][-4000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_11old['csh']['PotEng'][-4000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_11old['peg']['PotEng'][-4000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")


pe_npt1 = results_21old['close']['PotEng'][-4000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_31old['close']['PotEng'][-4000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 4000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11old), ("2:1", results_21old), ("3:1", results_31old)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True)
for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):
        ax = axes[i, j]        
        
        # Data
        pe = np.asarray(results[phase]["PotEng"][-nsteps:])
        x = np.arange(len(pe))
        
        # Raw trajectory
        ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
        
        # Moving average
        pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
        x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
        ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

        # Piecewise linear fits        
        slopes = []
        for start in range(0, len(pe) - fit_window + 1, fit_window):
            xs = x[start:start + fit_window]
            ys = pe[start:start + fit_window]
            z = np.polyfit(xs, ys, 1)
            p = np.poly1d(z)
            slopes.append(z[0])
            ax.plot(xs, p(xs), 'k--', lw=2)

        # # Summary statistics        
        # avg_slope = np.mean(slopes)
        # final_slope = slopes[-1]
        # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

        slope_text = ", ".join([f"{s:.3f}" for s in slopes])
        ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

        if i == 2: ax.set_xlabel("Time (steps)")
        if j == 0: ax.set_ylabel("PE (kcal/mol)")
        ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

systems = [
    ("1:1", results_11old),
    ("2:1", results_21old),
    ("3:1", results_31old),
]

phases = ["close", "peg", "csh"]

fig, axes = plt.subplots(3, 3, figsize=(15, 12), sharex=True)

for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):

        ax = axes[i, j]

        pe = results[phase]["PotEng"][-1000:]
        x = np.arange(len(pe))

        # Linear fit
        z = np.polyfit(x, pe, 1)
        p = np.poly1d(z)

        # Plot data and fit
        ax.plot(x, pe, 'r', lw=1)
        ax.plot(x, p(x), 'k--', lw=2)

        ax.set_title(f"{phase.upper()} ({ratio})\nSlope = {z[0]:.4f}",fontsize=10)

        if i == 2:
            ax.set_xlabel("Time (steps)")
        if j == 0:
            ax.set_ylabel("PE (kcal/mol)")

plt.tight_layout()
plt.show()

In [ ]:
pe_npt1 = results_11old['close']['PotEng'][-4000:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"S1: Slope of PE vs time: {z[0]:.6f} kcal/mol*step")
plt.xlabel("Time (steps)")
plt.ylabel("Potential Energy (kcal/mol)")
plt.show()

pe_npt1 = results_11old['csh']['PotEng'][-4000:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"S3: Slope of PE vs time: {z[0]:.6f} kcal/mol*step")
plt.xlabel("Time (steps)")
plt.ylabel("Potential Energy (kcal/mol)")
plt.show()

pe_npt1 = results_11old['peg']['PotEng'][-4000:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"S2: Slope of PE vs time: {z[0]:.6f} kcal/mol*step")
plt.xlabel("Time (steps)")
plt.ylabel("Potential Energy (kcal/mol)")
plt.show()



pe_npt1 = results_21old['close']['PotEng'][-4000:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"S1: Slope of PE vs time: {z[0]:.6f} kcal/mol*step")
plt.xlabel("Time (steps)")
plt.ylabel("Potential Energy (kcal/mol)")
plt.show()

pe_npt1 = results_31old['close']['PotEng'][-4000:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"S1: Slope of PE vs time: {z[0]:.6f} kcal/mol*step")
plt.xlabel("Time (steps)")
plt.ylabel("Potential Energy (kcal/mol)")
plt.show()


In [ ]:
pe_npt1 = results_11old['close']['PotEng'][-1000:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"S1: Slope of PE vs time: {z[0]:.6f} kcal/mol*step")
plt.xlabel("Time (steps)")
plt.ylabel("Potential Energy (kcal/mol)")
plt.show()

pe_npt1 = results_11old['csh']['PotEng'][-1000:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"S3: Slope of PE vs time: {z[0]:.6f} kcal/mol*step")
plt.xlabel("Time (steps)")
plt.ylabel("Potential Energy (kcal/mol)")
plt.show()

pe_npt1 = results_11old['peg']['PotEng'][-1000:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"S2: Slope of PE vs time: {z[0]:.6f} kcal/mol*step")
plt.xlabel("Time (steps)")
plt.ylabel("Potential Energy (kcal/mol)")
plt.show()



pe_npt1 = results_21old['close']['PotEng'][-1000:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"S1: Slope of PE vs time: {z[0]:.6f} kcal/mol*step")
plt.xlabel("Time (steps)")
plt.ylabel("Potential Energy (kcal/mol)")
plt.show()

pe_npt1 = results_31old['close']['PotEng'][-1000:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"S1: Slope of PE vs time: {z[0]:.6f} kcal/mol*step")
plt.xlabel("Time (steps)")
plt.ylabel("Potential Energy (kcal/mol)")
plt.show()


In [ ]:
pe_npt1 = results_11old['close']['PotEng'][5500:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_11old['csh']['PotEng'][5500:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_11old['peg']['PotEng'][5500:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")


pe_npt1 = results_11old['close']['PotEng'][5500:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")
plt.show()

pe_npt1 = results_11old['csh']['PotEng'][5500:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")
plt.show()

pe_npt1 = results_11old['peg']['PotEng'][5500:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
plt.title(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")
plt.show()

In [ ]:
RUN = "0_run10"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


In [ ]:
pe_npt1 = results_11n['csh']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_11n['peg']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_21n['peg']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_31n['peg']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_11n['close']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_21n['close']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_31n['close']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

systems = [
    ("1:1", results_11n),
    ("2:1", results_21n),
    ("3:1", results_31n),
]

phases = ["close", "peg", "csh"]

fig, axes = plt.subplots(3, 3, figsize=(15, 12), sharex=True)

for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):

        ax = axes[i, j]

        pe = results[phase]["PotEng"][-2000:]
        x = np.arange(len(pe))

        # Linear fit
        z = np.polyfit(x, pe, 1)
        p = np.poly1d(z)

        # Plot data and fit
        ax.plot(x, pe, 'r', lw=1)
        ax.plot(x, p(x), 'k--', lw=2)

        ax.set_title(f"{phase.upper()} ({ratio})\nSlope = {z[0]:.4f}",fontsize=10)

        if i == 2:
            ax.set_xlabel("Time (steps)")
        if j == 0:
            ax.set_ylabel("PE (kcal/mol)")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 4000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True)
for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):
        ax = axes[i, j]        
        
        # Data
        pe = np.asarray(results[phase]["PotEng"][-nsteps:])
        x = np.arange(len(pe))
        
        # Raw trajectory
        ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
        
        # Moving average
        pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
        x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
        ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

        # Piecewise linear fits        
        slopes = []
        for start in range(0, len(pe) - fit_window + 1, fit_window):
            xs = x[start:start + fit_window]
            ys = pe[start:start + fit_window]
            z = np.polyfit(xs, ys, 1)
            p = np.poly1d(z)
            slopes.append(z[0])
            ax.plot(xs, p(xs), 'k--', lw=2)

        # # Summary statistics        
        # avg_slope = np.mean(slopes)
        # final_slope = slopes[-1]
        # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

        slope_text = ", ".join([f"{s:.3f}" for s in slopes])
        ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

        if i == 2: ax.set_xlabel("Time (steps)")
        if j == 0: ax.set_ylabel("PE (kcal/mol)")
        ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
RUN = "0_run11"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


In [ ]:
pe_npt1 = results_11n['csh']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_11n['peg']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_21n['peg']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_31n['peg']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_11n['close']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_21n['close']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_31n['close']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 3000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True)
for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):
        ax = axes[i, j]        
        
        # Data
        pe = np.asarray(results[phase]["PotEng"][-nsteps:])
        x = np.arange(len(pe))
        
        # Raw trajectory
        ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
        
        # Moving average
        pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
        x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
        ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

        # Piecewise linear fits        
        slopes = []
        for start in range(0, len(pe) - fit_window + 1, fit_window):
            xs = x[start:start + fit_window]
            ys = pe[start:start + fit_window]
            z = np.polyfit(xs, ys, 1)
            p = np.poly1d(z)
            slopes.append(z[0])
            ax.plot(xs, p(xs), 'k--', lw=2)

        # # Summary statistics        
        # avg_slope = np.mean(slopes)
        # final_slope = slopes[-1]
        # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

        slope_text = ", ".join([f"{s:.3f}" for s in slopes])
        ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

        if i == 2: ax.set_xlabel("Time (steps)")
        if j == 0: ax.set_ylabel("PE (kcal/mol)")
        ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 3000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][-nsteps:])
    pe2 = np.asarray(results[phases[1]]["PotEng"][-nsteps:])
    pe3 = np.asarray(results[phases[2]]["PotEng"][-nsteps:])
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

    # Piecewise linear fits        
    slopes = []
    for start in range(0, len(pe) - fit_window + 1, fit_window):
        xs = x[start:start + fit_window]
        ys = pe[start:start + fit_window]
        z = np.polyfit(xs, ys, 1)
        p = np.poly1d(z)
        slopes.append(z[0])
        ax.plot(xs, p(xs), 'k--', lw=2)

    # # Summary statistics        
    # avg_slope = np.mean(slopes)
    # final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# ###### last 10
# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1038016.780000
# PEG + Water      : -72363.113200
# CSH + Water      : -965747.788000
# Separated System : -1038110.901200
# ------------------------------
# Adsorption Energy: 94.121200 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1040272.210000
# PEG + Water      : -74615.443400
# CSH + Water      : -965747.788000
# Separated System : -1040363.231400
# ------------------------------
# Adsorption Energy: 91.021400 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1041700.290000
# PEG + Water      : -75692.293600
# CSH + Water      : -965747.788000
# Separated System : -1041440.081600
# ------------------------------
# Adsorption Energy: -260.208400 kcal/mol
# ==============================




# ###### last 100

# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1038137.302000
# PEG + Water      : -72349.151700
# CSH + Water      : -965753.746000
# Separated System : -1038102.897700
# ------------------------------
# Adsorption Energy: -34.404300 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1040214.262000
# PEG + Water      : -74594.949120
# CSH + Water      : -965753.746000
# Separated System : -1040348.695120
# ------------------------------
# Adsorption Energy: 134.433120 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1041625.384000
# PEG + Water      : -75731.135010
# CSH + Water      : -965753.746000
# Separated System : -1041484.881010
# ------------------------------
# Adsorption Energy: -140.502990 kcal/mol
# ==============================


# # last 500
# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1038091.537400
# PEG + Water      : -72346.851272
# CSH + Water      : -965732.518540
# Separated System : -1038079.369812
# ------------------------------
# Adsorption Energy: -12.167588 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1040158.576800
# PEG + Water      : -74600.541232
# CSH + Water      : -965732.518540
# Separated System : -1040333.059772
# ------------------------------
# Adsorption Energy: 174.482972 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1041604.613400
# PEG + Water      : -75721.612336
# CSH + Water      : -965732.518540
# Separated System : -1041454.130876
# ------------------------------
# Adsorption Energy: -150.482524 kcal/mol
# ==============================



# # last 1000

# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1038052.625300
# PEG + Water      : -72349.848263
# CSH + Water      : -965703.972920
# Separated System : -1038053.821183
# ------------------------------
# Adsorption Energy: 1.195883 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1040126.716600
# PEG + Water      : -74600.893075
# CSH + Water      : -965703.972920
# Separated System : -1040304.865995
# ------------------------------
# Adsorption Energy: 178.149395 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1041572.457100
# PEG + Water      : -75726.672740
# CSH + Water      : -965703.972920
# Separated System : -1041430.645660
# ------------------------------
# Adsorption Energy: -141.811440 kcal/mol
# ==============================

In [ ]:
RUN = "0_run12_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()

avg_peg11 = np.mean(results_11n['peg']['PotEng'][-1000:])
avg_peg21 = np.mean(results_21n['peg']['PotEng'][-1000:])
avg_peg31 = np.mean(results_31n['peg']['PotEng'][-1000:])
print(f"Average PEG PotEng for 1:1: {avg_peg11:.4f}, 2:1: {avg_peg21:.4f}, 3:1: {avg_peg31:.4f}")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 2000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 500       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True)
for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):
        ax = axes[i, j]        
        
        # Data
        pe = np.asarray(results[phase]["PotEng"][-nsteps:])
        x = np.arange(len(pe))
        
        # Raw trajectory
        ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
        
        # Moving average
        pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
        x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
        ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

        # Piecewise linear fits        
        slopes = []
        for start in range(0, len(pe) - fit_window + 1, fit_window):
            xs = x[start:start + fit_window]
            ys = pe[start:start + fit_window]
            z = np.polyfit(xs, ys, 1)
            p = np.poly1d(z)
            slopes.append(z[0])
            ax.plot(xs, p(xs), 'k--', lw=2)

        # # Summary statistics        
        # avg_slope = np.mean(slopes)
        # final_slope = slopes[-1]
        # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

        slope_text = ", ".join([f"{s:.3f}" for s in slopes])
        ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

        if i == 2: ax.set_xlabel("Time (steps)")
        if j == 0: ax.set_ylabel("PE (kcal/mol)")
        ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 7000          # number of points from the end
smooth_window = 500    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][-nsteps:])
    pe2 = np.asarray(results[phases[1]]["PotEng"][-nsteps:])
    pe3 = np.asarray(results[phases[2]]["PotEng"][-nsteps:])
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

    # # Piecewise linear fits        
    # slopes = []
    # for start in range(0, len(pe) - fit_window + 1, fit_window):
    #     xs = x[start:start + fit_window]
    #     ys = pe[start:start + fit_window]
    #     z = np.polyfit(xs, ys, 1)
    #     p = np.poly1d(z)
    #     slopes.append(z[0])
    #     ax.plot(xs, p(xs), 'k--', lw=2)

    # # Summary statistics        
    # avg_slope = np.mean(slopes)
    # final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    # slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    slope_text = ''
    ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 7000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][-nsteps:])
    pe2 = np.asarray(results[phases[1]]["PotEng"][-nsteps:])
    pe3 = np.asarray(results[phases[2]]["PotEng"][-nsteps:])
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

    # Piecewise linear fits        
    slopes = []
    for start in range(0, len(pe) - fit_window + 1, fit_window):
        xs = x[start:start + fit_window]
        ys = pe[start:start + fit_window]
        z = np.polyfit(xs, ys, 1)
        p = np.poly1d(z)
        slopes.append(z[0])
        ax.plot(xs, p(xs), 'k--', lw=2)

    # # Summary statistics        
    # avg_slope = np.mean(slopes)
    # final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
RUN = "0_run12_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log_full.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log_full.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=5000)


# results_11n['peg'] has total length of 7000 while others are 12000 so we need to append the average of all property from last 1000 steps and make it 12000
for key in results_11n['peg'].keys():
    avg_value = np.mean(results_11n['peg'][key][-1000:])
    padding = np.full(5000, avg_value)
    results_11n['peg'][key] = np.concatenate([results_11n['peg'][key], padding])


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))

plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 9000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 3000       # local linear fit window

systems = [("1:1", results_11n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)

for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):
        ax = axes[j]
        
        # Data
        pe = np.asarray(results[phase]["PotEng"][-nsteps:])
        x = np.arange(len(pe))
        
        # Raw trajectory
        ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
        
        # Moving average
        pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
        x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
        ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

        # Piecewise linear fits        
        slopes = []
        for start in range(0, len(pe) - fit_window + 1, fit_window):
            xs = x[start:start + fit_window]
            ys = pe[start:start + fit_window]
            z = np.polyfit(xs, ys, 1)
            p = np.poly1d(z)
            slopes.append(z[0])
            ax.plot(xs, p(xs), 'k--', lw=2)

        # # Summary statistics        
        # avg_slope = np.mean(slopes)
        # final_slope = slopes[-1]
        # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

        slope_text = ", ".join([f"{s:.3f}" for s in slopes])
        ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

        ax.set_xlabel("Time (steps)")
        ax.set_ylabel("PE (kcal/mol)")
        ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 11000          # number of points from the end
smooth_window = 500    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(1, 1, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    # ax = axes[i]
    ax = axes
    
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][-nsteps:])
    pe2 = np.asarray(results[phases[1]]["PotEng"][-nsteps:])
    pe3 = np.asarray(results[phases[2]]["PotEng"][-nsteps:])
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

    # Piecewise linear fits        
    slopes = []
    # for start in range(0, len(pe) - fit_window + 1, fit_window):
    #     xs = x[start:start + fit_window]
    #     ys = pe[start:start + fit_window]
    #     z = np.polyfit(xs, ys, 1)
    #     p = np.poly1d(z)
    #     slopes.append(z[0])
    #     ax.plot(xs, p(xs), 'k--', lw=2)

    # # Summary statistics        
    # avg_slope = np.mean(slopes)
    # final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    # slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    slope_text = ''
    ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)
    ax.set_ylim([-1000,1000])
    plt.axhline(0)


# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
RUN = "0_run13_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 2000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 500       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True)
for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):
        ax = axes[i, j]        
        
        # Data
        pe = np.asarray(results[phase]["PotEng"][-nsteps:])
        x = np.arange(len(pe))
        
        # Raw trajectory
        ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
        
        # Moving average
        pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
        x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
        ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

        # Piecewise linear fits        
        slopes = []
        for start in range(0, len(pe) - fit_window + 1, fit_window):
            xs = x[start:start + fit_window]
            ys = pe[start:start + fit_window]
            z = np.polyfit(xs, ys, 1)
            p = np.poly1d(z)
            slopes.append(z[0])
            ax.plot(xs, p(xs), 'k--', lw=2)

        # # Summary statistics        
        # avg_slope = np.mean(slopes)
        # final_slope = slopes[-1]
        # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

        slope_text = ", ".join([f"{s:.3f}" for s in slopes])
        ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

        if i == 2: ax.set_xlabel("Time (steps)")
        if j == 0: ax.set_ylabel("PE (kcal/mol)")
        ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 7000          # number of points from the end
smooth_window = 500    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][-nsteps:])
    pe2 = np.asarray(results[phases[1]]["PotEng"][-nsteps:])
    pe3 = np.asarray(results[phases[2]]["PotEng"][-nsteps:])
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

    # # Piecewise linear fits        
    # slopes = []
    # for start in range(0, len(pe) - fit_window + 1, fit_window):
    #     xs = x[start:start + fit_window]
    #     ys = pe[start:start + fit_window]
    #     z = np.polyfit(xs, ys, 1)
    #     p = np.poly1d(z)
    #     slopes.append(z[0])
    #     ax.plot(xs, p(xs), 'k--', lw=2)

    # # Summary statistics        
    # avg_slope = np.mean(slopes)
    # final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    # slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    slope_text = ''
    ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:

RUN = "0_run8"

close_log = BASE / RUN / "1_all3/new_3ns_fixcom/log.lammps"
peg_log   = BASE / RUN / "2_peg/log.lammps"
csh_log   = BASE / RUN / "3_csh_water/new_3ns_fixcom/log.lammps"

# ============================================================
# RUN ANALYSIS
# ============================================================

results = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=2000000, average_last_n=500)


# ============================================================
# PLOT
# ============================================================

plot_thermo_grid(results)

In [ ]:
pe_npt1 = results['close']['PotEng'][2500:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

In [ ]:
pe_npt1 = results['peg']['PotEng'][2500:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

In [ ]:
pe_npt1 = results['csh']['PotEng'][2500:]
plt.plot(pe_npt1,'r')
# fit a line fitting the pe_npt1 and plot it
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
p = np.poly1d(z)
plt.plot(p(np.arange(len(pe_npt1))), 'k--')
# print slop of the line
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

In [ ]:
plt.plot(results['close']['Lz'][-900:])
plt.plot(results['peg']['Lz'][-900:]+results['csh']['Lz'][-900:])

plt.axhline(results['close']['Lz'][-900:].mean(), color='red', linestyle='--', label='mean system1')
plt.axhline((results['peg']['Lz'][-900:]+results['csh']['Lz'][-900:]).mean(), color='green', linestyle='--', label='mean system2+ system3)')
plt.legend()
plt.xlabel("npt step (1 ns total)")
plt.ylabel("Lz")

print(results['close']['Lz'][-900:].mean())
print(results['peg']['Lz'][-900:].mean())
print(results['csh']['Lz'][-900:].mean())
print((results['peg']['Lz'][-900:]+results['csh']['Lz'][-900:]).mean())

In [ ]:
plt.plot(results['close']['PotEng'][-900:])
plt.plot(results['peg']['PotEng'][-900:]+results['csh']['PotEng'][-900:])

plt.axhline(results['close']['PotEng'][-900:].mean(), color='red', linestyle='--', label='mean system1')
plt.axhline((results['peg']['PotEng'][-900:]+results['csh']['PotEng'][-900:]).mean(), color='green', linestyle='--', label='mean system2+ system3)')
plt.legend()
plt.xlabel("npt step (1 ns total)")
plt.ylabel("PotEng")

print(results['close']['PotEng'][-900:].mean())
print(results['peg']['PotEng'][-900:].mean())
print(results['csh']['PotEng'][-900:].mean())
print((results['peg']['PotEng'][-900:]+results['csh']['PotEng'][-900:]).mean())

In [ ]:
plt.plot(results['close']['PotEng'][:],'r')
plt.plot(results['peg']['PotEng'][:]+results['csh']['PotEng'][:],'b')

In [ ]:
RUN = "0_run7"

close_log = BASE / "0_run8/1_all3/log.lammps"
peg_log   = BASE / RUN / "2_peg/npt/log.lammps"
csh_log   = BASE / RUN / "3_csh_water/npt/log.lammps"


# ============================================================
# RUN ANALYSIS
# ============================================================

results_npt = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=1, average_last_n=1)


# ============================================================
# PLOT
# ============================================================

plot_thermo_grid(results_npt)

In [ ]:
# combined results dictionary
combined_results = {
    "close": {},
    "peg": {},
    "csh": {},
    'E_ads': None,
}

for key in results:
    if key in ['E_ads']:
        combined_results[key] = results[key]
    elif key in ['close']:
        for key1 in results[key]:
            combined_results[key][key1] = results_npt[key][key1]
    else:
        for key1 in results[key]:
            combined_results[key][key1] = np.concatenate([results[key][key1], results_npt[key][key1]])


plot_thermo_grid(combined_results)

In [ ]:
plt.plot(combined_results['close']['Lz'][-900:])
plt.plot(combined_results['peg']['Lz'][-900:]+combined_results['csh']['Lz'][-900:])

plt.axhline(combined_results['close']['Lz'][-900:].mean(), color='red', linestyle='--', label='mean system1')
plt.axhline((combined_results['peg']['Lz'][-900:]+combined_results['csh']['Lz'][-900:]).mean(), color='green', linestyle='--', label='mean system2+ system3)')
plt.legend()
plt.xlabel("npt step (1 ns total)")
plt.ylabel("Lz")

print(combined_results['close']['Lz'][-900:].mean())
print(combined_results['peg']['Lz'][-900:].mean())
print(combined_results['csh']['Lz'][-900:].mean())
print((combined_results['peg']['Lz'][-900:]+combined_results['csh']['Lz'][-900:]).mean())

In [ ]:
plt.plot(combined_results['close']['Lz'][-3000:])
plt.plot(combined_results['peg']['Lz'][-3000:]+combined_results['csh']['Lz'][-3000:])

plt.axhline(combined_results['close']['Lz'][-3000:].mean(), color='red', linestyle='--', label='mean system1')
plt.axhline((combined_results['peg']['Lz'][-3000:]+combined_results['csh']['Lz'][-3000:]).mean(), color='green', linestyle='--', label='mean system2+ system3)')
plt.legend()
plt.xlabel("npt step (1 ns total)")
plt.ylabel("Lz")

In [ ]:
plt.plot(combined_results['close']['Lz'][-900:])
plt.plot(combined_results['peg']['Lz'][-1900:-1000]+combined_results['csh']['Lz'][-1900:-1000])

plt.axhline(combined_results['close']['Lz'][-900:].mean(), color='red', linestyle='--', label='mean system1')
plt.axhline((combined_results['peg']['Lz'][-1900:-1000]+combined_results['csh']['Lz'][-1900:-1000]).mean(), color='green', linestyle='--', label='mean system2+ system3)')
plt.legend()
plt.xlabel("npt step (1 ns total)")
plt.ylabel("Lz")

In [ ]:
combined_results['close']

In [ ]:
plt.plot(combined_results['close']['PotEng'][-900:])
plt.plot(combined_results['peg']['PotEng'][-1900:-1000]+combined_results['csh']['PotEng'][-1900:-1000])

plt.axhline(combined_results['close']['PotEng'][-900:].mean(), color='red', linestyle='--', label='mean system1')
plt.axhline((combined_results['peg']['PotEng'][-1900:-1000]+combined_results['csh']['PotEng'][-1900:-1000]).mean(), color='green', linestyle='--', label='mean system2+ system3)')
plt.legend()
plt.xlabel("npt step (1 ns total)")
plt.ylabel("PotEng")

In [ ]:
combined_results['peg']['PotEng'][:]+combined_results['csh']['PotEng'][:]

In [ ]:
plt.plot(combined_results['close']['PotEng'][10:],'r')
plt.plot(combined_results['peg']['PotEng'][-6005:]+combined_results['csh']['PotEng'][-6005:],'b')

In [ ]:
plt.plot(combined_results['close']['PotEng'][-3000:])
plt.plot(combined_results['peg']['PotEng'][-6000:-1000]+combined_results['csh']['PotEng'][-6000:-1000])

plt.axhline(combined_results['close']['PotEng'][-3000:].mean(), color='red', linestyle='--', label='mean system1')
plt.axhline((combined_results['peg']['PotEng'][-6000:-1000]+combined_results['csh']['PotEng'][-6000:-1000]).mean(), color='green', linestyle='--', label='mean system2+ system3)')
plt.legend()
plt.xlabel("npt step (1 ns total)")
plt.ylabel("PotEng")

In [ ]:
print(sum(combined_results['close']['Lx']!=45.0592))
print(sum(combined_results['close']['Ly']!=59.4))

In [ ]:
# ss = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1) # 430
# ss = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=10) # 840
# ss = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100) # 1035
dd = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000) # 978
# ff = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=2000) # 2648

In [ ]:
RUN = "0_run6"

close_log = BASE / RUN / "1_all3/log.lammps"
peg_log   = BASE / RUN / "2_peg/log.lammps"
csh_log   = BASE / RUN / "3_csh_water/log.lammps"


# ============================================================
# RUN ANALYSIS
# ============================================================

results = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)

# ============================================================
# PLOT
# ============================================================

plot_thermo_grid(results)


In [ ]:
RUN = "0_run1/1_1-12E"

close_log1 = BASE / RUN / "close/log.lammps"
peg_log1   = BASE / RUN / "away1/log.lammps"
csh_log1   = BASE / RUN / "away1/log.lammps"

rr = compute_adsorption_energy(close_log=close_log1, peg_log=peg_log1, csh_log=csh_log1, cutoff_step=100, average_last_n=None, two_box=True)
results = compute_adsorption_energy(close_log=close_log1, peg_log=peg_log1, csh_log=csh_log1, cutoff_step=100, average_last_n=500, two_box=True)

plot_thermo_grid(results)

In [ ]:
RUN = "0_run1/2_1-8E"

close_log1 = BASE / RUN / "close/log.lammps"
peg_log1   = BASE / RUN / "away1/log.lammps"
csh_log1   = BASE / RUN / "away1/log.lammps"

results = compute_adsorption_energy(close_log=close_log1, peg_log=peg_log1, csh_log=csh_log1, cutoff_step=100, average_last_n=500, two_box=True)

plot_thermo_grid(results)

In [ ]:
RUN = "0_run1/3_1-6E"

close_log1 = BASE / RUN / "close/log.lammps"
peg_log1   = BASE / RUN / "away1/log.lammps"
csh_log1   = BASE / RUN / "away1/log.lammps"

results = compute_adsorption_energy(close_log=close_log1, peg_log=peg_log1, csh_log=csh_log1, cutoff_step=100, average_last_n=500, two_box=True)

plot_thermo_grid(results)